In [2]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import platform
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import psutil
import tensorflow as tf
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.preprocessing import MinMaxScaler, StandardScaler

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

TARGET_COL = "attack_cat"
N_REPEATS = 3


In [3]:
# Carga de dataset y preprocesado, igual que en los cuadernos de ataques para UNSW-NB15
path_train = "../../DATASETS/dataSets_Reducidos/nusw-nb15/datos_train_NUSW_redux.csv"
path_test = "../../DATASETS/dataSets_Reducidos/nusw-nb15/datos_test_NUSW_redux.csv"

df_train = pl.read_csv(path_train)
df_test = pl.read_csv(path_test)

y_train = (
    df_train.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

y_test = (
    df_test.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

x_train = df_train.drop(TARGET_COL)
x_test = df_test.drop(TARGET_COL)

X_train_raw = x_train.to_numpy().astype(np.float32)
X_test_raw = x_test.to_numpy().astype(np.float32)
X_full_raw = np.vstack([X_train_raw, X_test_raw]).astype(np.float32)

y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()
y_full_np = np.concatenate([y_train_np, y_test_np])

mlp_scaler = StandardScaler()
X_train_scaled_mlp = mlp_scaler.fit_transform(X_train_raw).astype(np.float32)
X_test_scaled_mlp = mlp_scaler.transform(X_test_raw).astype(np.float32)
X_full_scaled_mlp = mlp_scaler.transform(X_full_raw).astype(np.float32)

cnn_scaler = MinMaxScaler()
X_train_scaled_cnn = cnn_scaler.fit_transform(X_train_raw).astype(np.float32)
X_test_scaled_cnn = cnn_scaler.transform(X_test_raw).astype(np.float32)
X_full_scaled_cnn = cnn_scaler.transform(X_full_raw).astype(np.float32)

X_train_cnn = X_train_scaled_cnn.reshape(X_train_scaled_cnn.shape[0], X_train_scaled_cnn.shape[1], 1)
X_test_cnn = X_test_scaled_cnn.reshape(X_test_scaled_cnn.shape[0], X_test_scaled_cnn.shape[1], 1)
X_full_cnn = X_full_scaled_cnn.reshape(X_full_scaled_cnn.shape[0], X_full_scaled_cnn.shape[1], 1)


In [4]:
MODEL_INPUTS = {
    "test": {
        "rf": X_test_raw,
        "xgb": X_test_raw,
        "lgbm": X_test_raw,
        "catboost": X_test_raw,
        "svm": X_test_scaled_mlp,
        "mlp": X_test_scaled_mlp,
        "cnn": X_test_cnn,
    },
    "full": {
        "rf": X_full_raw,
        "xgb": X_full_raw,
        "lgbm": X_full_raw,
        "catboost": X_full_raw,
        "svm": X_full_scaled_mlp,
        "mlp": X_full_scaled_mlp,
        "cnn": X_full_cnn,
    },
}


In [5]:
# Tamaño de los datasets

N_MUESTRAS = len(X_full_raw)

print(N_MUESTRAS)

257673


In [6]:
# RF

rf_path = "../model/unsw-nb15/rf_unsw.joblib"

rf_model = joblib.load(rf_path)
print("Modelo Cargado")

# Warm-up
_ = rf_model.predict(X_full_raw[:min(1000, len(X_full_raw))])

# Medida Latencia
tiempos_rf = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = rf_model.predict(X_full_raw)
    t1 = time.perf_counter()
    tiempos_rf.append(t1 - t0)

tiempo_total_rf = float(np.mean(tiempos_rf))
throughput_rf = N_MUESTRAS / tiempo_total_rf

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_rf]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_rf:.4f} s")
print(f"Throughput aproximado: {throughput_rf:,.0f} muestras/s")

Modelo Cargado
Tiempos medidos: [0.078, 0.0652, 0.0646]
Tiempo medio de inferencia sobre todo el dataset: 0.0692 s
Throughput aproximado: 3,721,179 muestras/s


In [7]:
# XGBOOST

xgb_path = "../model/unsw-nb15/xgb_unsw.joblib"

xgb_model = joblib.load(xgb_path)
print("Modelo Cargado")

# Warm-up
_ = xgb_model.predict(X_full_raw[:min(1000, len(X_full_raw))])

# Medida Latencia
tiempos_xgb = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = xgb_model.predict(X_full_raw)
    t1 = time.perf_counter()
    tiempos_xgb.append(t1 - t0)

tiempo_total_xgb = float(np.mean(tiempos_xgb))
throughput_xgb = N_MUESTRAS / tiempo_total_xgb

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_xgb]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_xgb:.4f} s")
print(f"Throughput aproximado: {throughput_xgb:,.0f} muestras/s")

Modelo Cargado
Tiempos medidos: [0.0258, 0.0257, 0.0264]
Tiempo medio de inferencia sobre todo el dataset: 0.0260 s
Throughput aproximado: 9,916,876 muestras/s


/usr/lib/python3.11/pickle.py:1718: UserWarning: [11:34:50] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:402: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.11/pickle.py:1718: UserWarning: [11:34:50] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.11/pickle.py:1718: UserWarning: [11:34:50] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


In [8]:
# LIGHTGBM

lgbm_path = "../model/unsw-nb15/lgbm_unsw.joblib"

lgbm_model = joblib.load(lgbm_path)
print("Modelo Cargado")

# Warm-up
_ = lgbm_model.predict(X_full_raw[:min(1000, len(X_full_raw))])

# Medida Latencia
tiempos_lgbm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = lgbm_model.predict(X_full_raw)
    t1 = time.perf_counter()
    tiempos_lgbm.append(t1 - t0)

tiempo_total_lgbm = float(np.mean(tiempos_lgbm))
throughput_lgbm = N_MUESTRAS / tiempo_total_lgbm

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_lgbm]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_lgbm:.4f} s")
print(f"Throughput aproximado: {throughput_lgbm:,.0f} muestras/s")

/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Modelo Cargado
Tiempos medidos: [0.051, 0.0455, 0.0453]
Tiempo medio de inferencia sobre todo el dataset: 0.0473 s
Throughput aproximado: 5,452,214 muestras/s


/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [9]:
# CATBOOST

catboost_path = "../model/unsw-nb15/catboost_unsw.joblib"

catboost_model = joblib.load(catboost_path)
print("Modelo Cargado")

# Warm-up
_ = catboost_model.predict(X_full_raw[:min(1000, len(X_full_raw))])

# Medida Latencia
tiempos_catboost = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = catboost_model.predict(X_full_raw)
    t1 = time.perf_counter()
    tiempos_catboost.append(t1 - t0)

tiempo_total_catboost = float(np.mean(tiempos_catboost))
throughput_catboost = N_MUESTRAS / tiempo_total_catboost

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_catboost]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_catboost:.4f} s")
print(f"Throughput aproximado: {throughput_catboost:,.0f} muestras/s")



Modelo Cargado


Tiempos medidos: [0.1078, 0.0736, 0.0713]
Tiempo medio de inferencia sobre todo el dataset: 0.0842 s
Throughput aproximado: 3,058,951 muestras/s


In [10]:
# SVM

svm_path = "../model/unsw-nb15/svm_unsw.joblib"

svm_model = joblib.load(svm_path)
print("Modelo Cargado")

# Warm-up
_ = svm_model.predict(X_full_scaled_mlp[:min(1000, len(X_full_scaled_mlp))])

# Medida Latencia
tiempos_svm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = svm_model.predict(X_full_scaled_mlp)
    t1 = time.perf_counter()
    tiempos_svm.append(t1 - t0)

tiempo_total_svm = float(np.mean(tiempos_svm))
throughput_svm = N_MUESTRAS / tiempo_total_svm

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_svm]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_svm:.4f} s")
print(f"Throughput aproximado: {throughput_svm:,.0f} muestras/s")

Modelo Cargado
Tiempos medidos: [0.0108, 0.0088, 0.0052]
Tiempo medio de inferencia sobre todo el dataset: 0.0082 s
Throughput aproximado: 31,271,516 muestras/s


In [14]:
# MLP

mlp_path = "../model/unsw-nb15/mlp_unsw.joblib"

mlp_model = joblib.load(mlp_path)
print("Modelo Cargado")

# Warm-up
_ = mlp_model.predict(X_full_scaled_mlp[:min(1000, len(X_full_scaled_mlp))], batch_size=4096, verbose=0)

# Medida Latencia
tiempos_mlp = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = mlp_model.predict(X_full_scaled_mlp, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_mlp.append(t1 - t0)

tiempo_total_mlp = float(np.mean(tiempos_mlp))
throughput_mlp = N_MUESTRAS / tiempo_total_mlp

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_mlp]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_mlp:.4f} s")
print(f"Throughput aproximado: {throughput_mlp:,.0f} muestras/s")

Modelo Cargado
Tiempos medidos: [0.3478, 0.1478, 0.1489]
Tiempo medio de inferencia sobre todo el dataset: 0.2148 s
Throughput aproximado: 1,199,516 muestras/s


In [12]:
# CNN

cnn_path = "../model/unsw-nb15/cnn_unsw.joblib"

cnn_model = joblib.load(cnn_path)
print("Modelo Cargado")

# Warm-up
_ = cnn_model.predict(X_full_cnn[:min(1000, len(X_full_cnn))], batch_size=4096, verbose=0)

# Medida Latencia
tiempos_cnn = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = cnn_model.predict(X_full_cnn, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_cnn.append(t1 - t0)

tiempo_total_cnn = float(np.mean(tiempos_cnn))
throughput_cnn = N_MUESTRAS / tiempo_total_cnn

print(f"Tiempos medidos: {[round(t, 4) for t in tiempos_cnn]}")
print(f"Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_cnn:.4f} s")
print(f"Throughput aproximado: {throughput_cnn:,.0f} muestras/s")


Modelo Cargado
Tiempos medidos: [0.6033, 0.3121, 0.2803]
Tiempo medio de inferencia sobre todo el dataset: 0.3986 s
Throughput aproximado: 646,466 muestras/s


In [15]:
# Tabla comparativa final

tabla_comparativa = pd.DataFrame(
    [
        {"Modelo": "RF", "Tiempo medio (s)": tiempo_total_rf, "Muestras/s aprox": throughput_rf},
        {"Modelo": "XGBoost", "Tiempo medio (s)": tiempo_total_xgb, "Muestras/s aprox": throughput_xgb},
        {"Modelo": "LightGBM", "Tiempo medio (s)": tiempo_total_lgbm, "Muestras/s aprox": throughput_lgbm},
        {"Modelo": "CatBoost", "Tiempo medio (s)": tiempo_total_catboost, "Muestras/s aprox": throughput_catboost},
        {"Modelo": "SVM", "Tiempo medio (s)": tiempo_total_svm, "Muestras/s aprox": throughput_svm},
        {"Modelo": "MLP", "Tiempo medio (s)": tiempo_total_mlp, "Muestras/s aprox": throughput_mlp},
        {"Modelo": "CNN", "Tiempo medio (s)": tiempo_total_cnn, "Muestras/s aprox": throughput_cnn},
    ]
).sort_values("Tiempo medio (s)").reset_index(drop=True)

tabla_comparativa["Tiempo medio (s)"] = tabla_comparativa["Tiempo medio (s)"].round(4)
tabla_comparativa["Muestras/s aprox"] = tabla_comparativa["Muestras/s aprox"].round(0).astype(int)

display(tabla_comparativa)


,Modelo,Tiempo medio (s),Muestras/s aprox
0,SVM,0.0082,31271516
1,XGBoost,0.0260,9916876
2,LightGBM,0.0473,5452214
3,RF,0.0692,3721179
4,CatBoost,0.0842,3058951
5,MLP,0.2148,1199516
6,CNN,0.3986,646466
